In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.xgboost
import xgboost as xgb
from sklearn.metrics import (f1_score, matthews_corrcoef,
                             average_precision_score, roc_auc_score)
import json
import warnings
warnings.filterwarnings('ignore')

# Set tracking URI — lưu local
mlflow.set_tracking_uri('file:../mlruns')
mlflow.set_experiment('fraud-detection')

train = pd.read_csv('../data/train_final.csv')
test  = pd.read_csv('../data/test_final.csv')

X_train = train.drop(columns=['isFraud'])
y_train = train['isFraud']
X_test  = test.drop(columns=['isFraud'])
y_test  = test['isFraud']

# Load best threshold
with open('../models/best_threshold.json') as f:
    best_threshold = json.load(f)['threshold']

print(f"MLflow OK!")
print(f"Best threshold: {best_threshold}")

2026/04/01 15:45:52 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection' does not exist. Creating a new experiment.


MLflow OK!
Best threshold: 0.1


In [2]:
# Danh sách các model và kết quả đã có
experiments = [
    {
        'name': 'Logistic Regression',
        'params': {'model_type': 'logistic_regression', 'max_iter': 1000},
        'metrics': {'F1': 0.1074, 'MCC': 0.0851, 'AUPRC': 0.0830, 'AUC': 0.6414}
    },
    {
        'name': 'Decision Tree',
        'params': {'model_type': 'decision_tree', 'max_depth': 10},
        'metrics': {'F1': 0.4421, 'MCC': 0.4238, 'AUPRC': 0.2236, 'AUC': 0.7415}
    },
    {
        'name': 'LSTM',
        'params': {'model_type': 'lstm', 'epochs': 10, 'hidden_size': 64},
        'metrics': {'F1': 0.2388, 'MCC': 0.2156, 'AUPRC': 0.2229, 'AUC': 0.6694}
    },
    {
        'name': 'XGBoost Default',
        'params': {'model_type': 'xgboost', 'n_estimators': 200, 'max_depth': 6},
        'metrics': {'F1': 0.3579, 'MCC': 0.4487, 'AUPRC': 0.5649, 'AUC': 0.8812}
    },
    {
        'name': 'LightGBM',
        'params': {'model_type': 'lightgbm', 'n_estimators': 200, 'max_depth': 6},
        'metrics': {'F1': 0.3958, 'MCC': 0.4894, 'AUPRC': 0.5574, 'AUC': 0.8543}
    },
    {
        'name': 'XGBoost Tuned',
        'params': {'model_type': 'xgboost_tuned', 'threshold': best_threshold},
        'metrics': {'F1': 0.50, 'MCC': 0.5269, 'AUPRC': 0.5813, 'AUC': 0.8918}
    }
]

for exp in experiments:
    with mlflow.start_run(run_name=exp['name']):
        mlflow.log_params(exp['params'])
        mlflow.log_metrics(exp['metrics'])
    print(f"✅ Logged: {exp['name']}")

print("\nTất cả experiments đã được log!")

✅ Logged: Logistic Regression
✅ Logged: Decision Tree
✅ Logged: LSTM
✅ Logged: XGBoost Default
✅ Logged: LightGBM
✅ Logged: XGBoost Tuned

Tất cả experiments đã được log!


In [3]:
best_model = xgb.XGBClassifier()
best_model.load_model('../models/xgb_tuned.json')

with mlflow.start_run(run_name='XGBoost Tuned - Final'):
    # Log params
    mlflow.log_params({
        'threshold': best_threshold,
        'smote': True,
        'temporal_split': True,
        'feature_engineering': True
    })
    
    # Log metrics
    y_prob = best_model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= best_threshold).astype(int)
    
    mlflow.log_metrics({
        'F1':    round(f1_score(y_test, y_pred), 4),
        'MCC':   round(matthews_corrcoef(y_test, y_pred), 4),
        'AUPRC': round(average_precision_score(y_test, y_prob), 4),
        'AUC':   round(roc_auc_score(y_test, y_prob), 4)
    })
    
    # Log model
    mlflow.xgboost.log_model(best_model, 'xgb_tuned_model')
    
    # Log artifacts
    mlflow.log_artifact('../reports/baseline_comparison.png')
    mlflow.log_artifact('../reports/smote_comparison.png')
    mlflow.log_artifact('../reports/threshold_tuning.png')
    
    print("✅ Logged final model với artifacts!")

2026/04/01 15:45:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✅ Logged final model với artifacts!


In [5]:
print("="*50)
print("Để xem MLflow UI, mở terminal và gõ:")
print("="*50)
print()
print("  cd D:\\fraud_detection")
print("  mlflow ui")
print()
print("Sau đó mở trình duyệt vào: http://127.0.0.1:5000")
print()
print("Bạn sẽ thấy bảng so sánh tất cả 6 experiments!")

Để xem MLflow UI, mở terminal và gõ:

  cd D:\fraud_detection
  mlflow ui

Sau đó mở trình duyệt vào: http://127.0.0.1:5000

Bạn sẽ thấy bảng so sánh tất cả 6 experiments!
